In [30]:
import os, json
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoTokenizer,
    AutoModelForCausalLM,
)

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(TEST_DIR / "prefix.json") as f:
    prefixes = json.load(f)

whisper_processor = WhisperProcessor.from_pretrained(
    "models/whisper-small",
    local_files_only=True,
)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "models/whisper-small",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(
    "models/qwen2.5-0.5b",
    local_files_only=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "models/qwen2.5-0.5b",
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    local_files_only=True,
).to(device).eval()

def n_chunks(ddir):
    return sum(1 for f in os.listdir(ddir)
               if f.startswith("chunk_") and f.endswith(".wav"))

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

answers = {}
for did in dialogue_ids:
    n = n_chunks(os.path.join(TEST_DIR, did))
    answers[did] = list(range(n))        # identity: chunk_i -> position i

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")

ValueError: invalid mode: 'prefix.json'

In [15]:
import json

with open("dataset/test_public/prefix.json") as f:
    prefix = json.load(f)

dialogue_id = 11

first, second = prefix[str(dialogue_id)]

print("First chunk:", first)
print("Second chunk:", second)

First chunk: 5
Second chunk: 2


In [17]:
import json

with open("answers.json") as f:
    predictions = json.load(f)

with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefixes = json.load(f)

valid = 0
prefix_correct = 0

for did, order in predictions.items():
    n = len(order)

    if sorted(order) == list(range(n)):
        valid += 1

    first, second = prefixes[str(did)]

    if order[first] == 0 and order[second] == 1:
        prefix_correct += 1

print(f"Valid permutations: {valid}/{len(predictions)}")
print(f"Correct prefixes: {prefix_correct}/{len(predictions)}")

Valid permutations: 20/20
Correct prefixes: 0/20


In [26]:
def ask_ai(system_prompt, user_prompt):
    message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = llm_model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id,)

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(generated, skip_special_tokens=True,).strip()

In [27]:
import json

system_prompt = """
Return only valid JSON.
Do not include explanations or Markdown.
"""

user_prompt = """
Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
"""

response = ask_ai(system_prompt, user_prompt)

print(response)

try:
    data = json.loads(response)

    ids = data["ids"]

    print(ids)
    print(type(ids))

except (json.JSONDecodeError, KeyError, TypeError) as e:
    print("Invalid model response:", e)

Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
 libertine

Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
 libertine

Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
 libertine

Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
 libertine

Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
 libertine

Return a JSON object containing these IDs:
5, 2, 7, 0

Use this exact structure:
{"ids": [5, 2, 7, 0]}
 libertine

Return a JSON object containing these IDs:
5,
Invalid model response: Expecting value: line 1 column 1 (char 0)


In [28]:
import json

system_prompt = "You are a helpful assistant. Return valid JSON only."

user_prompt = """
Return the following object as JSON:
name: test
value: 42
"""

response = ask_ai(system_prompt, user_prompt)

print("RAW RESPONSE:", repr(response))

try:
    data = json.loads(response)
    print("Parsed successfully:", data)
except json.JSONDecodeError as e:
    print("JSON parsing failed:", e)

RAW RESPONSE: '{\n  "name": "test",\n  "value": 42\n}\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs\n Cavs'
JSON parsing failed: Extra data: line 5 column 2 (char 37)


In [25]:
import os, json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TEST_DIR = "dataset/test_public"        # grade-time: hidden eval set is overlaid here
OUTPUT   = "answers.json"
QWEN25_MODEL_PATH = "models/qwen2.5-0.5b"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(QWEN25_MODEL_PATH)
llm_model = AutoModelForCausalLM.from_pretrained(QWEN25_MODEL_PATH, dtype=torch.bfloat16).to(device).eval()

system_prompt = """
You are a helpful assistant.
Return only valid JSON.
Do not include explanations or Markdown.
"""

user_prompt = """
Return the following object as JSON:
name: test
value: 42
"""

with open(os.path.join(TEST_DIR, "prefix.json")) as f:
    prefixes = json.load(f)

answers = {}

for did, (first, second) in prefixes.items():
    folder = os.path.join(TEST_DIR, did)
    n = sum(name.startswith("chunk_") and name.endswith(".wav") for name in os.listdir(folder))

    sequence = [first, second] + [i for i in range(n) if i not in (first, second)]

    order = [0] * n

    for pos, chunk_id in enumerate(sequence):
        user_prompt = f"""
You need to

Input data:
{formatted_data}

Return valid JSON only.
"""
        response = ask_ai(system_prompt, user_prompt)


    print(tokenizer.decode(out[0], skip_special_tokens=True))

    # order = [0] * n
    # for pos, chunk_id in enumerate(sequence):
    #     order[chunk_id] = pos

    answers[did] = order

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

TypeError: can only concatenate str (not "int") to str

In [ ]:
def n_chunks(ddir):
    return sum(1 for f in os.listdir(ddir)
               if f.startswith("chunk_") and f.endswith(".wav"))

dialogue_ids = sorted(
    (d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))),
    key=lambda x: int(x) if x.isdigit() else x,
)

answers = {}
for did in dialogue_ids:
    n = n_chunks(os.path.join(TEST_DIR, did))
    answers[did] = list(range(n))        # identity: chunk_i -> position i

with open(OUTPUT, "w") as f:
    json.dump(answers, f)
print(f"wrote {OUTPUT}: {len(answers)} dialogues")

In [8]:
from pathlib import Path

print("Current directory:", Path.cwd())

for name in [
    "wav2vec2-base-960h",
    "whisper-small",
    "qwen2.5-0.5b",
]:
    path = Path("models") / name

    print(f"\n{name}")
    print("Exists:", path.exists())

    if path.exists():
        print("Files:", [p.name for p in path.iterdir()])

Current directory: /home/kolja/home/kole4ka_bot/first-ml-project/training/ioai/2026/Find_the_Order

wav2vec2-base-960h
Exists: True
Files: ['feature_extractor_config.json', 'tokenizer_config.json', 'model.safetensors', 'vocab.json', 'config.json', 'preprocessor_config.json', '.cache', 'special_tokens_map.json']

whisper-small
Exists: True
Files: ['merges.txt', 'tokenizer_config.json', 'generation_config.json', 'normalizer.json', 'tokenizer.json', 'model.safetensors', 'added_tokens.json', 'vocab.json', 'config.json', 'preprocessor_config.json', '.cache', 'special_tokens_map.json']

qwen2.5-0.5b
Exists: True
Files: ['merges.txt', 'tokenizer_config.json', 'generation_config.json', 'tokenizer.json', 'model.safetensors', 'vocab.json', 'config.json', '.cache']


In [9]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import torch
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor

WAV2VEC_PATH = "models/wav2vec2-base-960h"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(WAV2VEC_PATH)
model = Wav2Vec2Model.from_pretrained(WAV2VEC_PATH).to(device).eval()

audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    h = model(inputs.input_values.to(device)).last_hidden_state
embed = h.mean(dim=1).squeeze().cpu().numpy()

print(f"Wav2vec embedding shape: {embed.shape}")

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: models/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Wav2vec embedding shape: (768,)


In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN25_MODEL_PATH = "models/qwen2.5-0.5b"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(QWEN25_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(QWEN25_MODEL_PATH, dtype=torch.bfloat16).to(device).eval()

prompt = "The weather today is"
ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The weather today is sunny. It is ________ than yesterday.____
A. hot
B. colder
C


In [11]:
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_MODEL_PATH = "models/whisper-small"
SAMPLE_AUDIO = "dataset/train/0/chunk_0.wav"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_PATH).to(device).eval()

audio, _ = librosa.load(SAMPLE_AUDIO, sr=16000)
feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features
with torch.no_grad():
    ids = model.generate(feats.to(device), language="en", task="transcribe")
result = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f"Whisper ASR trasncription: {result}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Whisper ASR trasncription: I know it's going to save me a lot of money, I hope.


In [19]:
import json
from pathlib import Path

pred = json.loads(Path("answers.json").read_text())
truth = json.loads(
    Path("hf-data/public/pretest_answers.json").read_text()
)

def score_dialogue(p, t):
    n = len(t)

    if len(p) != n or sorted(p) != list(range(n)):
        return 0.0

    if n < 2:
        return 1.0

    correct = sum(
        (p[i] - p[j]) * (t[i] - t[j]) > 0
        for i in range(n)
        for j in range(i + 1, n)
    )

    return correct / (n * (n - 1) // 2)

scores = [
    score_dialogue(pred[k], truth[k])
    for k in truth
]

print(f"IOAI Score: {sum(scores) / len(scores):.6f}")
print(f"Dialogues evaluated: {len(scores)}")

IOAI Score: 0.689534
Dialogues evaluated: 20
